In [1]:
import pandas as pd

In [2]:
zona_cafetera_eventos=pd.read_csv("../data/cafe/zona_cafetera_eventos.csv")

In [3]:
zona_cafetera_eventos.head()

,id,fecha_evento,X,Y,Mes,rain_3d,rain_7d,rain_15d,rain_30d,heavy_rain_days,...,mean_precip,mean_temp,mean_humidity,temp_range,precip_variability,dry_days_before,slope_x_rain3d,mean_et0,total_et0,water_balance
0,18,2022-08-02,-75.564444,5.235556,8,74.5,198.0,484.2,928.2,15,...,30.940000,16.863333,86.533333,1.9,14.079692,0,0,3.343667,100.31,827.89
1,20,2022-07-31,-75.609722,5.248611,7,4.4,29.8,57.2,117.2,0,...,3.906667,20.083333,88.133333,2.5,3.535245,0,0,3.201000,96.03,21.17
2,21,2022-07-30,-75.547222,5.272222,7,95.8,194.1,379.8,797.2,11,...,26.573333,16.790000,90.633333,1.8,13.247262,0,0,3.039667,91.19,706.01
3,22,2022-07-30,-75.551667,5.276389,7,94.4,251.1,390.1,756.0,10,...,25.200000,16.400000,91.733333,2.6,14.456602,0,0,3.018667,90.56,665.44
4,23,2022-07-29,-75.492222,5.608056,7,61.4,173.3,327.2,654.4,6,...,21.813333,18.460000,89.666667,2.3,11.198504,0,0,3.259667,97.79,556.61


In [5]:
zona_cafetera_eventos.dtypes

Fecha            object
Departamento     object
Municipio        object
Y               float64
X               float64
id                int64
Mes               int64
dtype: object

In [11]:
eventos = pd.read_csv("../data/cafe/clima_30d.csv")

In [6]:
from datetime import datetime, timedelta
import requests


In [7]:
import sys
import os

# Añade la ruta donde está utils.py
sys.path.append(os.path.abspath("../scripts"))

from utils import obtener_clima_30d


In [10]:
archivo_salida = "clima_30d.csv"
ids_descargados = set()

# Si el archivo ya existe, cargamos los IDs que ya procesamos
if os.path.exists(archivo_salida):
    df_existente = pd.read_csv(archivo_salida)
    ids_descargados = set(df_existente['id'].unique())
else:
    df_existente = pd.DataFrame()

# ---------- Bucle para descargar datos ----------
guardados = 0

for i, row in zona_cafetera_eventos.iterrows():
    evento_id = row['id']
    if evento_id in ids_descargados:
        continue  # ya fue descargado

    lat = row['Y']
    lon = row['X']
    fecha_evento = row['Fecha']

    datos = obtener_clima_30d(lat, lon, fecha_evento)

    if datos:
        df_clima = pd.DataFrame(datos)
        df_clima["id"] = evento_id
        df_clima["fecha_evento"] = fecha_evento

        df_clima.to_csv(archivo_salida, mode='a', header=not os.path.exists(archivo_salida), index=False)
        guardados += 1

        if guardados % 100 == 0:
            print(f"Guardados {guardados} eventos")
    else:
        print(f"error con evento: {evento_id}")


Guardados 100 eventos
Guardados 200 eventos
Guardados 300 eventos
Guardados 400 eventos
Guardados 500 eventos


In [17]:
eventos.columns

Index(['fecha_clima', 'temperature_2m_mean', 'precipitation_sum',
       'relative_humidity_2m_mean', 'et0_fao_evapotranspiration', 'id',
       'fecha_evento'],
      dtype='object')

In [16]:
#Para eliminar columnas ACÁ:
eventos.drop(columns=['pressure_msl_mean'], inplace=True)

In [14]:
#Para renombrer columnas ACÁ:
eventos.rename(columns={
    'time': 'fecha_clima'}, 
               inplace=True)

In [20]:
#Para guardar ACÁ:
df_completo.to_csv("../data/cafe/zona_cafetera_eventos.csv", index=False)

In [ ]:
#Juntamos los datos de eventos con los datos climáticos descargados
df_completo = eventos.merge(zona_cafetera_eventos[['id', 'X', 'Y', 'Mes']], on='id', how='left')

In [21]:
from features import calcular_features

In [23]:
calcular_features(df_completo, "../data/cafe/zona_cafetera_eventos.csv")

Features extraídos para 2163 eventos
Columnas generadas: ['id', 'fecha_evento', 'X', 'Y', 'Mes', 'rain_3d', 'rain_7d', 'rain_15d', 'rain_30d', 'heavy_rain_days', 'max_daily_rain', 'soil_moisture_proxy', 'antecedent_moisture_x_maxrain', 'precip_temp_ratio', 'humidity_index', 'mean_precip', 'mean_temp', 'mean_humidity', 'temp_range', 'precip_variability', 'dry_days_before', 'slope_x_rain3d', 'mean_et0', 'total_et0', 'water_balance']


,id,fecha_evento,X,Y,Mes,rain_3d,rain_7d,rain_15d,rain_30d,heavy_rain_days,...,mean_precip,mean_temp,mean_humidity,temp_range,precip_variability,dry_days_before,slope_x_rain3d,mean_et0,total_et0,water_balance
0,18,2022-08-02,-75.564444,5.235556,8,74.5,198.0,484.2,928.2,15,...,30.940000,16.863333,86.533333,1.9,14.079692,0,0,3.343667,100.31,827.89
1,20,2022-07-31,-75.609722,5.248611,7,4.4,29.8,57.2,117.2,0,...,3.906667,20.083333,88.133333,2.5,3.535245,0,0,3.201000,96.03,21.17
2,21,2022-07-30,-75.547222,5.272222,7,95.8,194.1,379.8,797.2,11,...,26.573333,16.790000,90.633333,1.8,13.247262,0,0,3.039667,91.19,706.01
3,22,2022-07-30,-75.551667,5.276389,7,94.4,251.1,390.1,756.0,10,...,25.200000,16.400000,91.733333,2.6,14.456602,0,0,3.018667,90.56,665.44
4,23,2022-07-29,-75.492222,5.608056,7,61.4,173.3,327.2,654.4,6,...,21.813333,18.460000,89.666667,2.3,11.198504,0,0,3.259667,97.79,556.61
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2158,5624,2000-11-05,-76.388000,2.989600,11,15.9,77.4,207.4,362.0,1,...,12.066667,16.720000,83.866667,3.7,8.230526,0,0,2.678000,80.34,281.66
2159,5626,2000-10-31,-76.584033,2.441050,10,52.7,120.8,241.5,595.2,3,...,19.840000,16.433333,85.300000,2.5,9.270584,0,0,2.882000,86.46,508.74
2160,5627,2000-10-28,-75.397167,5.600722,10,64.2,128.8,228.8,373.8,2,...,12.460000,14.776667,86.333333,2.3,7.712656,0,0,3.069667,92.09,281.71
2161,5628,2000-10-21,-75.487436,7.661475,10,28.5,76.9,146.4,304.7,0,...,10.156667,24.820000,89.000000,2.0,4.845526,0,0,3.537333,106.12,198.58


In [24]:
eventos_neg=pd.read_csv("../data/cafe/zona_cafetera_eventos_neg.csv")

In [25]:
len(zona_cafetera_eventos),len(eventos_neg)

(2163, 2163)

In [26]:
eventos_neg.columns

Index(['id', 'fecha_evento', 'X', 'Y', 'Mes', 'rain_3d', 'rain_7d', 'rain_15d',
       'rain_30d', 'heavy_rain_days', 'max_daily_rain', 'soil_moisture_proxy',
       'antecedent_moisture_x_maxrain', 'precip_temp_ratio', 'humidity_index',
       'mean_precip', 'mean_temp', 'mean_humidity', 'temp_range',
       'precip_variability', 'dry_days_before', 'slope_x_rain3d', 'mean_et0',
       'total_et0', 'water_balance'],
      dtype='object')

In [9]:
zona_cafetera_eventos['Fecha'] = pd.to_datetime(zona_cafetera_eventos['Fecha'], errors='coerce')

In [ ]:
zona_cafetera_eventos['Mes'] = zona_cafetera_eventos['Fecha'].dt.month
import matplotlib.pyplot as plt
eventos_por_Mes = zona_cafetera_eventos['Mes'].value_counts().sort_index()
plt.figure(figsize=(10, 6))
eventos_por_Mes.plot(kind='bar', color='skyblue')
plt.title('Número de eventos por Mes')
plt.xlabel('Mes')
plt.ylabel('Número de eventos')

In [ ]:
#revisar cuantos eventos quedaron por Mes en un grafico de barras
eventos_neg['Mes'] = eventos_neg['Fecha'].dt.month
import matplotlib.pyplot as plt
eventos_por_Mes = eventos_neg['Mes'].value_counts().sort_index()
plt.figure(figsize=(10, 6))
eventos_por_Mes.plot(kind='bar', color='skyblue')
plt.title('Número de eventos por Mes')
plt.xlabel('Mes')
plt.ylabel('Número de eventos')

In [41]:
from pyproj import Transformer

# EPSG:3116 (metros) → EPSG:4326 (grados)
transformer = Transformer.from_crs("EPSG:9377", "EPSG:4326", always_xy=True)

lon, lat = transformer.transform(4722723.58129186,2117705.10252585)
print(f"Latitud: {lat}, Longitud: {lon}")


Latitud: 5.060439506181107, Longitud: -75.50171182725427
